<a href="https://colab.research.google.com/github/springboardmentor12458j/LiveMeetingSummarize/blob/Monica/Transcriptions8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q vosk gTTS faster-whisper pydub soundfile sentencepiece
!apt -qq install -y ffmpeg

In [ ]:
from google.colab import files
from IPython.display import Audio,display
import os, wave, json, subprocess
from gtts import gTTS
from vosk import Model,KaldiRecognizer
import torch
from faster_whisper import WhisperModel


print("Please upload files (or none to use generated text-to-speech audio): ")
up = files.upload()

input_audio_path = None
original_text = None # Renamed text_input to original_text for clarity

if up:
  input_audio_path = list(up.keys())[0]
  print(f"Using uploaded audio file: {input_audio_path}")
  display(Audio(input_audio_path))
else:
  original_text = "This audio is generated from text for testing purpose"
  tts = gTTS(text = original_text, lang='en')
  tts.save('tts.mp3')
  input_audio_path = 'tts.mp3'
  print("Generated audio from text:", original_text)
  display(Audio(input_audio_path))

# 2) Convert → WAV 16khz mono
wav_output_path = "audio_16k_mono.wav" # Using a more descriptive name

if input_audio_path: # Ensure an audio file path is available
  subprocess.run(["ffmpeg","-y","-i", input_audio_path, "-ar","16000","-ac","1", wav_output_path],
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
  print(f"Converted {input_audio_path} to {wav_output_path} (16k mono)")
  display(Audio(wav_output_path))
else:
  print("Error: No audio input path found for conversion.")


# 3) --- VOSK STT ---
if not os.path.exists("vosk-model"):
    !wget -q https://alphacephei.com/vosk/models/vosk-model-small-en-us-0.15.zip -O model.zip
    !unzip -q model.zip
    !mv vosk-model-small-en-us-0.15 vosk-model
    !rm model.zip

wf = wave.open(wav_output_path,'rb') # Use the correct WAV file for Vosk
rec = KaldiRecognizer(Model("vosk-model"),16000)

vosk_text = ""

while True:
  data = wf.readframes(4000)
  if not data: break
  if rec.AcceptWaveform(data):
    vosk_text+= json.loads(rec.Result()).get('text'," ")+ " "
vosk_text += json.loads(rec.FinalResult()).get('text'," ")+ " "


# 4) --- whisper stt---

device = 'cuda' if torch.cuda.is_available() else 'cpu'
whisper = WhisperModel('small',device=device)

segments,_ = whisper.transcribe(wav_output_path) # Use the correct WAV file for Whisper
whisper_text = " ".join([s.text for s in segments]).strip()

# 5) Results
print("\n===== VOSK OUTPUT =====")
print(vosk_text)

print("\n===== WHISPER OUTPUT =====")
print(whisper_text)

if original_text: # Only print original text if it was generated
  print("\n===== ORIGINAL TEXT =====")
  print(original_text)